# Blackboard Pattern | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, List, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class BlackboardState(TypedDict):
    patient_info: str
    blackboard: List[str]       # Shared contributions from all agents
    symptoms_analyzed: NotRequired[bool]  # Track which specialists have contributed
    labs_interpreted: NotRequired[bool]
    diagnosis_generated: NotRequired[bool]
    diagnosis: NotRequired[str]

In [5]:
def controller(state: BlackboardState) -> Command[Literal["symptom_analyst", "lab_interpreter", "diagnosis_generator", "finalize"]]:
    """Inspect blackboard STATE and decide based on what's missing — deterministic, not LLM-driven."""
    if not state.get("symptoms_analyzed"):
        choice = "symptom_analyst"
    elif not state.get("labs_interpreted"):
        choice = "lab_interpreter"
    elif not state.get("diagnosis_generated"):
        choice = "diagnosis_generator"
    else:
        choice = "finalize"
    print(f"[Controller] Blackboard state: symptoms={state.get('symptoms_analyzed', False)}, "
          f"labs={state.get('labs_interpreted', False)}, diagnosis={state.get('diagnosis_generated', False)} "
          f"-> routing to '{choice}'")
    return Command(goto=choice, update={})

def symptom_analyst(state: BlackboardState) -> dict:
    bb_content = "\n".join(state.get("blackboard", []))
    response = model.invoke(
        f"You are a symptom analyst. Analyze the patient's symptoms and add your findings to the blackboard.\n\n"
        f"Patient: {state['patient_info']}\n"
        f"Current blackboard:\n{bb_content or '(empty)'}\n\n"
        f"Provide: symptom patterns, severity assessment, possible systems affected."
    )
    new_entry = f"[Symptom Analyst]: {response.content}"
    return {"blackboard": state.get("blackboard", []) + [new_entry], "symptoms_analyzed": True}

def lab_interpreter(state: BlackboardState) -> dict:
    bb_content = "\n".join(state.get("blackboard", []))
    response = model.invoke(
        f"You are a lab results interpreter. Given the patient info and current findings, "
        f"interpret any lab values or suggest what labs to order.\n\n"
        f"Patient: {state['patient_info']}\n"
        f"Current blackboard:\n{bb_content or '(empty)'}\n\n"
        f"Provide: lab interpretations, reference ranges, abnormality flags."
    )
    new_entry = f"[Lab Interpreter]: {response.content}"
    return {"blackboard": state.get("blackboard", []) + [new_entry], "labs_interpreted": True}

def diagnosis_generator(state: BlackboardState) -> dict:
    bb_content = "\n".join(state.get("blackboard", []))
    response = model.invoke(
        f"You are a diagnosis generator. Based on all blackboard findings, generate differential diagnoses.\n\n"
        f"Patient: {state['patient_info']}\n"
        f"Current blackboard:\n{bb_content or '(empty)'}\n\n"
        f"Provide: ranked differential diagnoses with confidence levels and supporting evidence."
    )
    new_entry = f"[Diagnosis Generator]: {response.content}"
    return {"blackboard": state.get("blackboard", []) + [new_entry], "diagnosis_generated": True}

def finalize(state: BlackboardState) -> dict:
    bb_content = "\n".join(state.get("blackboard", []))
    response = model.invoke(
        f"Synthesize all blackboard findings into a final diagnostic summary.\n\n"
        f"Patient: {state['patient_info']}\n\n"
        f"Blackboard:\n{bb_content}\n\n"
        f"Provide: primary diagnosis, supporting evidence, recommended next steps, and confidence level."
    )
    return {"diagnosis": response.content}

In [6]:
graph = StateGraph(BlackboardState)
graph.add_node("controller", controller, destinations=("symptom_analyst", "lab_interpreter", "diagnosis_generator", "finalize"))
graph.add_node("symptom_analyst", symptom_analyst)
graph.add_node("lab_interpreter", lab_interpreter)
graph.add_node("diagnosis_generator", diagnosis_generator)
graph.add_node("finalize", finalize)

graph.add_edge(START, "controller")
# controller returns Command to route directly
graph.add_edge("symptom_analyst", "controller")
graph.add_edge("lab_interpreter", "controller")
graph.add_edge("diagnosis_generator", "controller")
graph.add_edge("finalize", END)

blackboard = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(blackboard)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	controller(controller)
	symptom_analyst(symptom_analyst)
	lab_interpreter(lab_interpreter)
	diagnosis_generator(diagnosis_generator)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> controller;
	controller -.-> diagnosis_generator;
	controller -.-> finalize;
	controller -.-> lab_interpreter;
	controller -.-> symptom_analyst;
	diagnosis_generator --> controller;
	lab_interpreter --> controller;
	symptom_analyst --> controller;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = blackboard.invoke({
    "patient_info": "45-year-old male, presenting with fatigue, unexplained weight loss (10 lbs in 2 months), "
                    "increased thirst and urination, blurred vision. Fasting glucose: 280 mg/dL, HbA1c: 9.2%.",
    "blackboard": [],
})
print(result["diagnosis"])

[Controller] Blackboard state: symptoms=False, labs=False, diagnosis=False -> routing to 'symptom_analyst'
[Controller] Blackboard state: symptoms=True, labs=False, diagnosis=False -> routing to 'lab_interpreter'
[Controller] Blackboard state: symptoms=True, labs=True, diagnosis=False -> routing to 'diagnosis_generator'
[Controller] Blackboard state: symptoms=True, labs=True, diagnosis=True -> routing to 'finalize'
### Diagnostic Summary

**Primary Diagnosis: Type 2 Diabetes Mellitus**

**Supporting Evidence:**
- **Symptoms Consistent with Diabetes:** The patient reports fatigue, unexplained weight loss, increased thirst, and urination, along with blurred vision. These are classic symptoms associated with diabetes, particularly type 2.
- **Laboratory Findings:** 
  - Fasting glucose level is significantly elevated at 280 mg/dL, where the normal range is typically 70-99 mg/dL, and diabetes is diagnosed at 126 mg/dL or higher.
  - HbA1c is 9.2%, indicating chronic hyperglycemia. A normal

In [9]:
stream_invoke(blackboard, {
    "patient_info": "45-year-old male, presenting with fatigue, unexplained weight loss (10 lbs in 2 months), "
                    "increased thirst and urination, blurred vision. Fasting glucose: 280 mg/dL, HbA1c: 9.2%.",
    "blackboard": [],
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

[Controller] Blackboard state: symptoms=False, labs=False, diagnosis=False -> routing to 'symptom_analyst'
[Controller] Blackboard state: symptoms=True, labs=False, diagnosis=False -> routing to 'lab_interpreter'
[Controller] Blackboard state: symptoms=True, labs=True, diagnosis=False -> routing to 'diagnosis_generator'
[Controller] Blackboard state: symptoms=True, labs=True, diagnosis=True -> routing to 'finalize'
────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'patient_info': '45-year-old male, presenting with fatigue, unexplained weight loss (10 lbs in 2 months), increased thirst and urination, blurred vision. Fasting glucose: 280 mg/dL, HbA1c: 9.2%.',
 'blackboard': ['[Symptom Analyst]: Current Blackboard Analysis:\n\n1. **Symptom Patterns:**\n   - **Fatigue**: Common in metabolic disorders; possibly due to lack of energy conversion from glucose.\n   - **Unexplained Weight Loss**: Suggestive of metabolic dysfunction; body breaking down fat/muscle due to inadequate glucose usage.\n   - **Increased Thirst and Urination**: Indicates possible osmotic diuresis due to high blood glucose levels.\n   - **Blurred Vision**: Could be caused by changes in lens shape due to fluctuating blood glucose levels.\n\n2. **Severity Assessment:**\n   - **Hyperglycemia**: Fasting glucose of 280 mg/dL is significantly elevated, indicating poor blood sugar control.\n   - **Elevated HbA1c**: At 9.2%, it shows average blood glucose over past 3 months is substantial